[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/corrections/seance2_correction.ipynb)

# Séance 4.2 — Prédire une décision — qui va résilier ?

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- transformer des colonnes de texte en variables utilisables par un modèle
- ajuster une régression logistique et lire une probabilité de départ
- lire une matrice de confusion et nommer les deux façons de se tromper
- calculer et interpréter justesse, précision, rappel et F1
- expliquer pourquoi la justesse est un piège sur des données déséquilibrées
- choisir un seuil de décision à partir d'un coût, pas d'une habitude

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")   ## texte -> nombre
tel = tel.dropna(subset=["total"])   ## 11 abonnes tout neufs, sans facture

y = tel["churn"]   ## 1 = il est parti, 0 = il est reste
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True)   ## texte -> 0/1

# stratify=y : le meme taux de resiliation des deux cotes du decoupage
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(len(X_tr), "abonnes d'apprentissage,", len(X_te), "de test")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le contrat, déjà

> **Votre mission :**
> - Calculer le taux de résiliation par type de contrat, en % arrondi à 1 décimale → `par_contrat`.
> - Mettre celui des contrats mensuels dans `taux_mensuel`.

In [ ]:
# mean() sur une colonne 0/1 = la part de 1, donc le taux de depart
par_contrat = (tel.groupby("contrat")["churn"].mean() * 100).round(1)
taux_mensuel = par_contrat["mensuel"]   ## 42,7 %

print(par_contrat)

# 42,7 % contre 2,8 % : un facteur quinze entre le contrat mensuel et
# l'engagement deux ans.

In [ ]:
verifier("1 - churn des contrats mensuels", taux_mensuel == 42.7,
         "groupby('contrat') puis mean() sur churn")

### Exercice 2 — Ajuster le modèle

> **Votre mission :**
> - Construire un pipeline `StandardScaler` puis `LogisticRegression(max_iter=1000)` → `m`, et l'ajuster sur l'apprentissage.
> - Récupérer les probabilités de départ du jeu de test → `proba`.

In [ ]:
# make_pipeline enchaine les etapes : mise a l'echelle, puis modele
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
m.fit(X_tr, y_tr)   ## sur l'apprentissage uniquement

# predict_proba renvoie deux colonnes : [proba de 0, proba de 1].
# Celle qui nous interesse est la seconde, l'indice 1.
proba = m.predict_proba(X_te)[:, 1]
print(proba[:5].round(3))

In [ ]:
verifier("2a - nombre de probabilites", len(proba) == 2110, "on predit sur le jeu de test")
verifier("2b - ce sont bien des probabilites", proba.min() >= 0 and proba.max() <= 1,
         "la colonne d'indice 1 est la probabilite de depart")

### Exercice 3 — Le piège de la justesse

> **Votre mission :**
> - Calculer la justesse du modèle sur le test → `just_modele` (en %, 1 décimale).
> - Puis celle d'un modèle qui prédit que **personne** ne part → `just_nul`.
> - Combien de points le modèle gagne-t-il réellement ?

In [ ]:
pred = m.predict(X_te)   ## des 0 et des 1, tranches a 0,50

just_modele = round(100 * accuracy_score(y_te, pred), 1)   ## 79,8 %

# La part de ceux qui restent : c'est la justesse du modele qui dit
# toujours "reste"
just_nul = round(100 * (1 - y_te.mean()), 1)
print(just_modele, "% contre", just_nul, "% ->", round(just_modele - just_nul, 1), "points")

In [ ]:
verifier("3a - justesse du modele", abs(just_modele - 79.8) < 0.5, "accuracy_score(y_te, pred)")
verifier("3b - justesse du modele nul", just_nul == 73.4,
         "c'est la proportion de clients qui restent")

### Exercice 4 — La matrice de confusion

> **Votre mission :**
> - Afficher la matrice de confusion du modèle.
> - Mettre dans `perdus` le nombre de clients **partis sans qu'on les ait détectés**.
> - C'est la case qui coûte de l'argent.

In [ ]:
mat = confusion_matrix(y_te, pred)
print(mat)   ## lignes : la verite | colonnes : la prediction

# Ligne 1 = ceux qui partent vraiment, colonne 0 = ceux qu'on predit
# comme restant. L'intersection : les partants qu'on n'a pas vus venir.
perdus = mat[1][0]
print(perdus, "clients perdus sans rien tenter")

In [ ]:
verifier("4 - clients perdus non detectes", abs(perdus - 255) <= 5,
         "ligne des vrais partants, colonne des predits restants")

### Exercice 5 — Précision, rappel et F1

> **Votre mission :**
> - Calculer la précision → `prec`, le rappel → `rapp` et le F1 → `f1` du modèle, arrondis à 3 décimales.
> - Vérifier que le F1 vaut bien `2 × prec × rapp / (prec + rapp)` → `f1_main`.
> - Traduire précision et rappel en une phrase de gestion, en commentaire.

In [ ]:
prec = round(precision_score(y_te, pred), 3)   ## parmi ceux qu'on appelle
rapp = round(recall_score(y_te, pred), 3)     ## parmi ceux qui partent
f1 = round(f1_score(y_te, pred), 3)           ## les deux en un seul nombre

# La formule du F1, refaite a la main : elle n'a rien de mysterieux
f1_main = round(2 * prec * rapp / (prec + rapp), 3)
print("precision", prec, "| rappel", rapp, "| F1", f1, "| a la main", f1_main)

# Precision 0,64 : sur 10 abonnes contactes, 6 allaient vraiment partir.
# Rappel 0,545 : nous retrouvons un partant sur deux, l'autre s'en va.

In [ ]:
verifier("5a - precision", abs(prec - 0.640) < 0.02, "precision_score")
verifier("5b - rappel", abs(rapp - 0.545) < 0.02, "la fonction s'appelle recall_score")
verifier("5c - F1", abs(f1 - 0.589) < 0.02, "f1_score(y_te, pred)")
verifier("5d - F1 refait a la main", abs(f1 - f1_main) < 0.01,
         "2 fois le produit, divise par la somme")

### Exercice 6 — Descendre le seuil

> **Votre mission :**
> - Décider à **0,30** au lieu de 0,50 → `pred30`.
> - Recalculer précision, rappel et F1 → `prec30`, `rapp30`, `f1_30`.
> - Lequel des deux premiers monte, lequel descend ? Et le F1 ?

In [ ]:
pred30 = (proba > 0.30).astype(int)   ## on appelle des qu'il y a 30 % de risque

prec30 = round(precision_score(y_te, pred30), 3)   ## descend
rapp30 = round(recall_score(y_te, pred30), 3)      ## monte
f1_30 = round(f1_score(y_te, pred30), 3)           ## bouge a peine
print("seuil 0.30 -> precision", prec30, "| rappel", rapp30, "| F1", f1_30)

# Le rappel monte (on retrouve plus de partants), la precision descend
# (on contacte plus de gens pour rien). C'est toujours ce compromis, et
# c'est pour ca que le F1, qui tient les deux, ne bouge presque pas.

In [ ]:
verifier("6a - rappel a 0,30", rapp30 > rapp, "un seuil plus bas retrouve plus de partants")
verifier("6b - precision a 0,30", prec30 < prec, "et contacte plus de monde pour rien")
verifier("6c - F1 a 0,30", abs(f1_30 - 0.610) < 0.02, "f1_score(y_te, pred30)")

### Exercice 7 — Le gain de la campagne

> **Votre mission :**
> - Un appel coûte **15 €**, un client retenu rapporte **300 €**, une relance en convainc **30 %**.
> - Calculer le gain au seuil 0,50 → `gain50`, puis au seuil 0,20 → `gain20`.

In [ ]:
def gain(seuil):
    p = (proba > seuil).astype(int)
    vrais = ((p == 1) & (y_te == 1)).sum()     ## partants effectivement rattrapes
    return vrais * 0.30 * 300 - p.sum() * 15   ## 15 euros par appel passe

gain50 = gain(0.50)
gain20 = gain(0.20)
print(round(gain50), "euros contre", round(gain20), "euros")

# 8 000 EUR d'ecart, pour un seul nombre change dans une comparaison.

In [ ]:
verifier("7a - gain au seuil 0,50", abs(gain50 - 20370) < 600, "le cout d'un appel est 15")
verifier("7b - gain au seuil 0,20", abs(gain20 - 28365) < 600, "meme calcul, seuil 0.20")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - Le directeur de la relation client vous demande combien d'appels prévoir et ce que ça rapporte.
> - Au seuil retenu de 0,20 : le nombre d'appels → `nb_appels`, et le gain par appel → `gain_appel` (arrondi à 2 décimales).
> - Puis rédigez votre recommandation en commentaire.

In [ ]:
p20 = (proba > 0.20).astype(int)

nb_appels = int(p20.sum())   ## sum() compte les 1
gain_appel = round(gain20 / nb_appels, 2)   ## ce que rapporte chaque appel
print(nb_appels, "appels |", gain_appel, "euros de gain par appel")

# Recommandation possible :
# "Sur 2 110 abonnes, le modele en designe 1 073 a rappeler. La campagne
#  rapporte 28 400 EUR nets, soit 26 EUR par appel passe. Le seuil de
#  0,20 est volontairement bas : nos appels coutent 15 EUR et un client
#  retenu en rapporte 300, il est donc rentable d'appeler large. Ce
#  reglage doit etre revu si le cout d'un appel augmente."

In [ ]:
verifier("8a - nombre d'appels", abs(nb_appels - 1073) < 40, "sum() compte les 1")
verifier("8b - gain par appel", abs(gain_appel - 26.4) < 2, "divisez le gain par le nombre d'appels")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Quand l'appel coûte plus cher

> **Votre mission :**
> - Refaire le calcul du gain avec un appel à **60 €** au lieu de 15 € — une visite commerciale plutôt qu'un appel.
> - Où passe l'optimum ? Qu'est-ce que ça dit du réglage d'un modèle ?

In [ ]:
def gain_cher(seuil, cout):
    p = (proba > seuil).astype(int)
    vrais = ((p == 1) & (y_te == 1)).sum()
    return vrais * 0.30 * 300 - p.sum() * cout

for cout in [15, 60]:
    g = pd.Series([gain_cher(s, cout) for s in np.arange(0.05, 1.0, 0.05)],
                  index=np.arange(0.05, 1.0, 0.05).round(2))
    print(f"cout {cout} EUR -> seuil optimal {g.idxmax()}, gain {round(g.max())} EUR")

# Quand le contact coute cher, il faut etre plus selectif : le seuil
# optimal remonte. Le meme modele, les memes probabilites, et pourtant
# une decision differente. Le modele ne decide pas, il informe.

### Question 10 — Les coefficients de la logistique

> **Votre mission :**
> - Extraire les coefficients du modèle et les trier par valeur absolue décroissante.
> - Les variables ayant été mises à l'échelle, ils sont comparables entre eux.
> - Quelles sont les trois qui pèsent le plus, et dans quel sens ?
> - *Nouveau :* dans un pipeline, on accède à la dernière étape par `m[-1]`.

In [ ]:
# m[-1] : la derniere etape du pipeline, la logistique elle-meme
coefs = pd.Series(m[-1].coef_[0], index=X.columns)

coefs.reindex(coefs.abs().sort_values(ascending=False).index).round(3).head(6)

# Un coefficient positif pousse au depart, negatif retient. On retrouve
# le contrat en tete — la seance 4.3 confirmera avec d'autres methodes.

### Question 11 — Le meilleur F1 est-il le meilleur seuil ?

> **Votre mission :**
> - Chercher le seuil qui maximise le **F1**, puis celui qui maximise le **gain** de la campagne.
> - Sont-ils au même endroit ? Combien coûte le fait de suivre le F1 plutôt que l'euro ?
> - *Nouveau :* `serie.idxmax()` donne l'indice de la plus grande valeur.

In [ ]:
seuils = np.arange(0.05, 1.0, 0.05).round(2)
f1s = pd.Series([f1_score(y_te, (proba > s).astype(int)) for s in seuils], index=seuils)
gains = pd.Series([gain(s) for s in seuils], index=seuils)

print("meilleur F1   : seuil", f1s.idxmax(), "->", round(f1s.max(), 3))
print("meilleur gain : seuil", gains.idxmax(), "->", round(gains.max()), "euros")
print("gain au seuil choisi par le F1 :", round(gain(f1s.idxmax())), "euros")

# Le F1 place l'optimum a 0,35, l'argent a 0,20. Suivre le F1 couterait
# 2 500 EUR. C'est normal et il faut le comprendre : le F1 traite les deux
# erreurs comme si elles avaient le meme poids, alors qu'ici un appel
# inutile coute 15 EUR et un client perdu en coute 90. Une mesure
# statistique cadre la decision, elle ne la prend pas.

### Question 12 — Un modèle plus simple fait-il pire ?

> **Votre mission :**
> - Ajuster un second modèle avec **trois variables seulement** : `anc`, `mensuel` et le contrat.
> - Comparer son F1 à celui du modèle complet.
> - La différence justifie-t-elle de collecter les onze autres colonnes ?

In [ ]:
petit = pd.get_dummies(tel[["anc", "mensuel", "contrat"]], drop_first=True)   ## 3 variables
a_tr, a_te, b_tr, b_te = train_test_split(
    petit, y, test_size=0.3, random_state=42, stratify=y)

simple = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
simple.fit(a_tr, b_tr)

print("modele complet :", round(f1_score(y_te, pred), 3))
print("modele a 3 var :", round(f1_score(b_te, simple.predict(a_te)), 3))

# 0,589 contre 0,548 : quatre centiemes de F1 pour onze colonnes de plus.
# Trois variables reproduisent donc l'essentiel — la question a poser
# AVANT de lancer un chantier de collecte de donnees.

### Question 13 — Le déséquilibre, traité autrement

> **Votre mission :**
> - Réajuster la logistique avec `class_weight='balanced'`, qui donne plus de poids à la classe rare.
> - Comparer justesse, précision, rappel et F1 au modèle d'origine.
> - En quoi cet effet ressemble-t-il à celui du seuil ?

In [ ]:
# class_weight="balanced" : la classe rare pese plus lourd a l'apprentissage
equilibre = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_tr, y_tr)
pe = equilibre.predict(X_te)

for nom, q in {"origine": pred, "equilibre": pe}.items():
    print(f"{nom:<10} justesse {accuracy_score(y_te, q):.3f}  "
          f"precision {precision_score(y_te, q):.3f}  rappel {recall_score(y_te, q):.3f}  "
          f"F1 {f1_score(y_te, q):.3f}")

# La justesse BAISSE (0,798 -> 0,725), le rappel monte fortement
# (0,545 -> 0,795), et le F1 monte legerement (0,589 -> 0,606). C'est le
# meme arbitrage qu'un seuil abaisse, obtenu cette fois pendant
# l'apprentissage. Deux chemins, une seule question : quelle erreur coute
# le plus cher ?

### Question 14 — Qui sont les faux positifs ?

> **Votre mission :**
> - Isoler les abonnés que le modèle prédit partants **à tort**.
> - Comparer leur profil (ancienneté, facture, contrat) à celui des vrais partants.
> - Ces appels sont-ils vraiment perdus ?

In [ ]:
test = X_te.copy()
test["reel"] = y_te
test["predit"] = pred

faux = test.query("predit == 1 and reel == 0")   ## appeles pour rien ?
vrais = test.query("predit == 1 and reel == 1")  ## appeles a raison

print("faux positifs :", len(faux), "| vrais positifs :", len(vrais))
colonnes = ["anc", "mensuel", "contrat_mensuel"]   ## on inclut le contrat
pd.DataFrame({"faux alerte": faux[colonnes].median(),
              "vrais partants": vrais[colonnes].median()}).round(1)

# Les faux positifs ressemblent beaucoup aux vrais partants : facture
# elevee (80 EUR des deux cotes), contrat mensuel dans 99 % des cas, et
# une anciennete a peine plus longue (9 mois contre 5). Ce ne sont pas des
# appels perdus, ce sont des clients A RISQUE qui ne sont pas encore
# partis.

### Question 15 — Question de synthèse

> **Votre mission :**
> - Le comité hésite : faut-il investir dans un meilleur modèle, ou dans un meilleur ciblage ?
> - Chiffrez les deux pistes : le gain obtenu en passant du seuil 0,50 au seuil optimal, et celui qu'apporterait un modèle parfait (rappel de 1 sans faux positifs) au seuil 0,50.
> - Puis tranchez, en commentaire.

In [ ]:
gains = pd.Series([gain(s) for s in np.arange(0.05, 1.0, 0.05)],
                  index=np.arange(0.05, 1.0, 0.05).round(2))

partants = int(y_te.sum())
parfait = partants * 0.30 * 300 - partants * 15   ## rappel 1, aucun faux positif

print("seuil 0,50           :", round(gain(0.50)), "euros")
print("meilleur seuil       :", round(gains.max()), "euros")
print("modele parfait       :", round(parfait), "euros")

# Regler le seuil rapporte 8 000 EUR et coute une apres-midi. Un modele
# PARFAIT — inatteignable — plafonnerait a 42 000 EUR. L'essentiel du
# gain disponible s'obtient donc par le ciblage, pas par l'algorithme.
# C'est la reponse a donner au comite.